# Train classification model

Imports

In [3]:
import torch
from torch import nn
import torchvision.transforms as transforms
import sys
import os
import torch.nn.functional as F
from ray.train import Checkpoint, get_checkpoint
from ray import tune

# Absolute path to your project's src folder
src_path = os.path.abspath(os.path.join("..", "src"))

# Add to sys.path if not already present
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(sys.path)

from data import get_data_loaders, show_image
from models.classification_model import ClassificationModel
from parameter_optimization import optimize_parameters, get_best_config
import config

['/Users/bevanslabbert/Documents/GitHub/pid-radast/src', '/Users/bevanslabbert/Documents/GitHub/pid-radast/.venv/lib/python3.13/site-packages/ray/thirdparty_files', '/opt/homebrew/Cellar/python@3.13/3.13.5/Frameworks/Python.framework/Versions/3.13/lib/python313.zip', '/opt/homebrew/Cellar/python@3.13/3.13.5/Frameworks/Python.framework/Versions/3.13/lib/python3.13', '/opt/homebrew/Cellar/python@3.13/3.13.5/Frameworks/Python.framework/Versions/3.13/lib/python3.13/lib-dynload', '', '/Users/bevanslabbert/Documents/GitHub/pid-radast/.venv/lib/python3.13/site-packages', '/var/folders/xl/d7wznkt15fq8wspgq66fyc9m0000gn/T/tmpieo8vn3g']


Data Initialization

In [4]:
trainloader, testloader = get_data_loaders(
    "Mirabest",
    transforms.Compose([
        transforms.ToTensor(), # to range [0,1]
        transforms.Normalize([0.5], [0.5]) # 0 centers
        ]),
        config.CLASSIFICATION_MODEL["batch_size"])

print(next(iter(trainloader))[0].shape)
print(next(iter(trainloader)))

Files already downloaded and verified
Files already downloaded and verified
torch.Size([2, 1, 150, 150])
[tensor([[[[-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          ...,
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.]]],


        [[[-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          ...,
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.]]]]), tensor([1, 1])]


Parameter optimization

In [5]:
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)

tuned_config = get_best_config(ClassificationModel, parent_dir)
print(tuned_config)
model = ClassificationModel(tuned_config)
criterion = tuned_config['criterion_class']()
optimizer = tuned_config['optimizer_class'](model.parameters(), lr=tuned_config['lr'])
batch_size = tuned_config['batch_size']

for epoch in range(config.CLASSIFICATION_MODEL["num_epochs"]):  # loop over the dataset multiple times

    print(f'Epoch {epoch+1}')
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # get the inputs
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()

print('Finished Training')



Getting config from /Users/bevanslabbert/Documents/GitHub/pid-radast/tuning_results/ClassificationModel
{'lr': 0.002631505273927559, 'optimizer_class': <class 'torch.optim.adam.Adam'>, 'model_class': <class 'models.classification_model.ClassificationModel'>, 'criterion_class': <class 'torch.nn.modules.loss.CrossEntropyLoss'>, 'dataset': 'MiraBest', 'batch_size': 16, 'kernel_size': 5, 'conv1_out_channels': 24, 'conv2_out_channels': 32, 'fc1_units': 84, 'fc2_units': 84}
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9
Epoch 10
Finished Training


In [6]:
# get data
dataiter = iter(testloader)
images, labels = next(dataiter)

# get predictions
output = model(images)
_, predicted = torch.max(output, 1)

In [7]:
correct = 0
total = 0
with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print('Accuracy of the network on the 88 test images: %d %%' % (100 * correct / total))

Accuracy of the network on the 88 test images: 68 %


In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adagrad(optimized_model.parameters(), lr=0.01)

for epoch in range(config.CLASSIFICATION_MODEL["num_epochs"]):  # loop over the dataset multiple times
    print(f'Epoch {epoch+1}')
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # get the inputs
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = optimized_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % print_num == (print_num-1):    # print every 50 mini-batches
            print('[%d, %5d] loss: %.3f' %
                  (epoch + 1, i + 1, running_loss / print_num))
            running_loss = 0.0

print('Finished Training')

NameError: name 'optimized_model' is not defined

In [ ]:
# get data
dataiter = iter(testloader)
images, labels = next(dataiter)

# get predictions
output = optimized_model(images)
_, predicted = torch.max(output, 1)

correct = 0
total = 0
with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = optimized_model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print('Accuracy of the network on the 88 test images: %d %%' % (100 * correct / total))


Accuracy of the network on the 88 test images: 74 %


Train

Test